# Setup Code

In [1]:
import os
import sys
from functools import partial
from pathlib import Path
from typing import Callable

import einops
import plotly.express as px
import plotly.graph_objects as go
import torch as t
from IPython.display import display
from ipywidgets import interact
from jaxtyping import Bool, Float
from torch import Tensor
from tqdm import tqdm

In [2]:
from arena_helpers.arena_utils import (
    render_lines_with_plotly,
    setup_widget_fig_ray,
    setup_widget_fig_triangle
)
sys.path.insert(0, './arena_helpers')
import arena_helpers.tests as tests

will make a helper function for plotting

# Rays & Segments

## Rays_1D

we'll make a function that generates some rays coming out of the origin. Which will take (0,0,0)

In [3]:
def make_rays_1d(num_pixels: int, y_limit: float) -> Tensor:
    """
    num_pixels: The number of pixels in the y dimension. Since there is one ray per pixel, this is
        also the number of rays.
    y_limit: At x=1, the rays should extend from -y_limit to +y_limit, inclusive of both endpoints.

    Returns: shape (num_pixels, num_points=2, num_dim=3) where the num_points dimension contains
        (origin, direction) and the num_dim dimension contains xyz.

    Example of make_rays_1d(9, 1.0): [
        [[0, 0, 0], [1, -1.0, 0]],
        [[0, 0, 0], [1, -0.75, 0]],
        [[0, 0, 0], [1, -0.5, 0]],
        ...
        [[0, 0, 0], [1, 0.75, 0]],
        [[0, 0, 0], [1, 1, 0]],
    ]
    """
    ## we'll use torch.linspace 
    y = t.linspace(-y_limit, y_limit, num_pixels)
    x= t.ones_like(y)
    z= t.zeros_like(y)
    ## stack columns
    result = t.stack([x,y,z], dim=1)
    ## add the origin points
    origin_points = t.zeros_like(result)
    return t.stack([origin_points, result], dim=1)


rays1d = make_rays_1d(9, 10.0)
fig = render_lines_with_plotly(rays1d)

## Ray-Object Intersection

Let's play with parameritization of the problem.

In [4]:
fig: go.FigureWidget = setup_widget_fig_ray()
display(fig)


@interact(v=(0.0, 6.0, 0.01), seed=(0, 10, 1))
def update(v=0.0, seed=0):
    t.manual_seed(seed)
    L_1, L_2 = t.rand(2, 2)
    P = lambda v: L_1 + v * (L_2 - L_1)
    x, y = zip(P(0), P(6))
    with fig.batch_update():
        fig.update_traces({"x": x, "y": y}, 0)
        fig.update_traces({"x": [L_1[0], L_2[0]], "y": [L_1[1], L_2[1]]}, 1)
        fig.update_traces({"x": [P(v)[0]], "y": [P(v)[1]]}, 2)

FigureWidget({
    'data': [{'type': 'scatter', 'uid': '536baad6-ae48-4338-a540-54f02f6fd98a', 'x': [], 'y': []},
             {'marker': {'size': 12},
              'mode': 'markers',
              'name': 'v=0',
              'type': 'scatter',
              'uid': '941c640f-0874-45ab-81cd-39d439196f4b',
              'x': [],
              'y': []},
             {'marker': {'size': 12, 'symbol': 'x'},
              'mode': 'markers',
              'name': 'v=1',
              'type': 'scatter',
              'uid': 'd88bca10-153f-4a77-88f2-cd52d8d2c1a9',
              'x': [],
              'y': []}],
    'layout': {'height': 400,
               'margin': {'b': 10, 'l': 40, 't': 60},
               'showlegend': False,
               'template': '...',
               'title': {'text': 'Ray coordinates illustration'},
               'width': 500,
               'xaxis': {'range': [-1.5, 2.5]},
               'yaxis': {'range': [-1.5, 2.5]}}
})

interactive(children=(FloatSlider(value=0.0, description='v', max=6.0, step=0.01), IntSlider(value=0, descript…

Suppose we have a line segment defined by points L1L1​ and L2L2​. Then for a given ray, we can test if the ray intersects the line segment like so:

- Supposing both the ray and line segment were infinitely long, solve for their intersection point.
- If the point exists, check whether that point is inside the line segment and the ray.


In [14]:
def intersect_ray_1d(ray: Float[Tensor, "points dims"], segment: Float[Tensor, "points dims"]) -> bool:
    """
    ray: shape (n_points=2, n_dim=3)  # O, D points
    segment: shape (n_points=2, n_dim=3)  # L_1, L_2 points

    Return True if the ray intersects the segment.
    """
    # extracting data
    l1, l2 = segment
    O, D = ray

    # only get x,y coordinates
    l1_l2 = (l1-l2)[0:2]
    D = D[0:2]
    l1_O = (l1-O)[0:2]

    mat_1 = t.stack([D, l1_l2], dim=1)
    if abs(t.linalg.det(mat_1)) > 1e-6:
        u_v = t.linalg.solve(mat_1, l1_O)
        v = u_v[1].item()
        u = u_v[0].item()
        return (v>=0 and v<=1) and (u >= 0)
    
    return False
    
tests.test_intersect_ray_1d(intersect_ray_1d)
tests.test_intersect_ray_1d_special_case(intersect_ray_1d)

All tests in `test_intersect_ray_1d` passed!
All tests in `test_intersect_ray_1d_special_case` passed!
